<a href="https://colab.research.google.com/github/josimardtm/ASP1/blob/main/optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Confiabilidad SAIDI & SAIFI
Edward Alejandro Ortiz gomez
Escuela colombiana de ingenieiria Julio Garavito
2022
INFINITY - Modelo de Optimización para, SAIDI, SAIFI, ENS y COSTO
"""
#==============================================================================
# IMPORTAR LIBRERIAS PYOMO
#==============================================================================
import pyomo.environ as pyo
from pyomo.opt import SolverFactory
#==============================================================================
# IMPORTAR LIBRERIAS MODELADO DE RED
#==============================================================================
import pandapower as pp
import pandapower.networks
import pandapower.topology
import pandapower.plotting
import pandapower.converter
import pandapower.estimation
import FunRed
import random
import time
inicio = time.time()
#==============================================================================
# IMPORTAR LIBRERIAS GENERALES
#==============================================================================
import pandas as pd
import numpy as np
import math as ma
#==============================================================================
# IMPORTAR RED
#==============================================================================
net=FunRed.creared()                #Importa la Red
print('Importación de Red completa')
pp.runpp(net)                       #Ejecuta Flujo de Carga
print('')
BfunS=FunRed.BusFunSwtch(net)       #Identificar el Bus en función del Switch
#==============================================================================
# IMPORTAR SCRIPT's
#==============================================================================
exec(open("GF_Geneador_de_fallas.py").read())
exec(open("GD_Generador_de_Demanda.py").read())
#==============================================================================
# DEFINICIÓN DE FUNCIONES
#==============================================================================
#FUNCIÓN -SelMesDeman- SELECCIONA EL MES DE LA DEMANDA BASADO EN EL DÍA DE FALLA
def SelMesDeman(Labr, Lmay, Ljun, Ljul, Lago, Lsep, Loct, DA):
    #Colht = pd.DataFrame()                 #Crea un DF temporal vacio
    Lmes=pd.DataFrame()
    if DA<=31:#ENERO
        Lmes=Labr
    elif DA>31 and DA<=59:#FEBRERO
        Lmes=Labr
    elif DA>59 and DA<=90:#MARZO
        Lmes=Labr
    elif DA>90 and DA<=120: # ABRIL
        Lmes=Labr
    elif DA>120 and DA<=151: #MAYO
        Lmes=Lmay
    elif DA>151 and DA<=181: #JUNIO
        Lmes=Ljun
    elif DA>181 and DA<=212: #JULIO
        Lmes=Ljul
    elif DA> 212 and DA<=243: #AGOSTO
        Lmes=Lago
    elif DA> 243 and DA<=273: #SEPTIEMBRE
        Lmes=Lsep
    elif DA> 273 and DA<=304: #OCTUBRE
        Lmes=Loct
    elif DA> 304 and DA<=334: #NOVIEMBRE
        Lmes=Loct
    elif DA> 334 and DA<=365: #DICIEOMBRE
        Lmes=Loct
    return(Lmes)

#FUNCIÓN -GenTimFall- ASIGNA EL DÍA, LA HORA, LA DURACIÓN DE LA FALLA EN FUNCIÓN DEL SWITCH Y LA ZONA
def GenTimFall_PU(EstiP,SwitchP,Zona1t,Zona2t,Zona3t):

    IdenZon=EstiP[EstiP['Proteccion']==SwitchP]   #Identificarzona del Switch (Filtra por nombre de Switches en la lista de Switches)
    IdenZon.index=[1]  #Renombrar el indice del dataframe
    if IdenZon.loc[1,'Zona']=='Z1':
        IdxFilale1=int(np.random.uniform(0,len(Zona1t),1))   #Indice de la fila alaeatoria de la Zona 1
        Dia_dl_Ano=Zona1t.loc[IdxFilale1,'Dia_dl_Ano']         #Crea Varable del Dia
        Hora_dl_Dia=Zona1t.loc[IdxFilale1,'Hora_dl_Dia']       #Crea Varable la hora
        tiempo_pr_fall=Zona1t.loc[IdxFilale1,'tiempo_pr_fall'] #Crea Varable del tiempo
        Coltm={'Proteccion':SwitchP,'Dia_dl_Ano':Dia_dl_Ano, 'Hora_dl_Dia':Hora_dl_Dia,'tiempo_pr_fall':tiempo_pr_fall} #Crea Columnas del nuevo Dataframe
        MusZon1=pd.DataFrame(data=Coltm,index=range(0,1)) #Crea el DataFrame de Salida
    elif IdenZon.loc[1,'Zona']=='Z2':
        IdxFilale2=int(np.random.uniform(0,len(Zona2t),1))   #Indice de la fila alaeatoria de la Zona 1
        Dia_dl_Ano=Zona2t.loc[IdxFilale2,'Dia_dl_Ano']         #Crea Varable del Dia
        Hora_dl_Dia=Zona2t.loc[IdxFilale2,'Hora_dl_Dia']       #Crea Varable la hora
        tiempo_pr_fall=Zona2t.loc[IdxFilale2,'tiempo_pr_fall'] #Crea Varable del tiempo
        Coltm={'Proteccion':SwitchP,'Dia_dl_Ano':Dia_dl_Ano, 'Hora_dl_Dia':Hora_dl_Dia,'tiempo_pr_fall':tiempo_pr_fall} #Crea Columnas del nuevo Dataframe
        MusZon1=pd.DataFrame(data=Coltm,index=range(0,1)) #Crea el DataFrame de Salida
    else:
        IdxFilale3=int(np.random.uniform(0,len(Zona3t),1))   #Indice de la fila alaeatoria de la Zona 1
        Dia_dl_Ano=Zona3t.loc[IdxFilale3,'Dia_dl_Ano']         #Crea Varable del Dia
        Hora_dl_Dia=Zona3t.loc[IdxFilale3,'Hora_dl_Dia']       #Crea Varable la hora
        tiempo_pr_fall=Zona3t.loc[IdxFilale3,'tiempo_pr_fall'] #Crea Varable del tiempo
        Coltm={'Proteccion':SwitchP,'Dia_dl_Ano':Dia_dl_Ano, 'Hora_dl_Dia':Hora_dl_Dia,'tiempo_pr_fall':tiempo_pr_fall} #Crea Columnas del nuevo Dataframe
        MusZon1=pd.DataFrame(data=Coltm,index=range(0,1)) #Crea el DataFrame de Salida
    return(MusZon1)

#FUNCIÓN - Eneryclient_PU- CALCULA LA ENS, TIEMPO Y NUMERO DE CLIENTES SIN SERVICIO, POTENCIA P Y Q  Y ESF PARA UN SOLO SWITCH
def Eneryclient_PU(Zona): #Función extare la columna estimada de demanda segun el mes y hora de falla
    print('================================================================================')
    print('Inicia Calculo de Modelo')
    print('================================================================================')
    print('Inicia')
    print('Procesando....')
    PSF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,3))        #Dataframe para guardar la potencia p en falla para cada protección
    QSF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,3))        #Dataframe para guardar la potencia Q en falla para cada protección
    LPSF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,3))       #Dataframe para guardar la potencia de perdidas p en falla para cada protección (a)
    LQSF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,3))       #Dataframe para guardar la potencia de perdidas Q en falla para cada protección (a)
    ESF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,3))        #Dataframe para guardar la energia en falla
    ESEN=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,3))       #Dataframe para guardar la energia en estado normal
    DIES=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,3))       #Diferencia de la energia
    SA1=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(len(Dload))) #Dataframe para almacenar los clientes sin servicio en cada falla
    DIF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,3))        #Diferencia de la energia
    #LOAD0=pd.DataFrame(columns=Zona.loc[:,'Proteccion'])
    #for j in range(len(Zona)):
    DA=Zona.iloc[0]['Dia_dl_Ano']          #Extrae el día de la falla
    HD=int(Zona.iloc[0]['Hora_dl_Dia'])    #Extrae la hora de la falla
    TF=Zona.iloc[0]['tiempo_pr_fall']      #Extrae el tiempo de la falla
    Colht = pd.DataFrame()                 #Crea un DF temporal vacio
    Lmes=pd.DataFrame()
    if DA<=31:#ENERO
        Lmes=Labr
    elif DA>31 and DA<=59:#FEBRERO
        Lmes=Labr
    elif DA>59 and DA<=90:#MARZO
        Lmes=Labr
    elif DA>90 and DA<=120: # ABRIL
        Lmes=Labr
    elif DA>120 and DA<=151: #MAYO
        Lmes=Lmay
    elif DA>151 and DA<=181: #JUNIO
        Lmes=Ljun
    elif DA>181 and DA<=212: #JULIO
        Lmes=Ljul
    elif DA> 212 and DA<=243: #AGOSTO
        Lmes=Lago
    elif DA> 243 and DA<=273: #SEPTIEMBRE
        Lmes=Lsep
    elif DA> 273 and DA<=304: #OCTUBRE
        Lmes=Loct
    elif DA> 304 and DA<=334: #NOVIEMBRE
        Lmes=Loct
    elif DA> 334 and DA<=365: #DICIEOMBRE
        Lmes=Loct
    #print("---------------------------------------")
    #print('->Falla ocurrida el día ' + str(DA))
    #print('->En la hora ' + str(HD))
    #print('->Durante ' + str(TF)+'Horas')
    #print("---------------------------------------")
    #print(Zona.iloc[0]['Proteccion'])
    clien=[]
    for k in range(ma.ceil(TF)):              #Repite dependiendo de el numero redondeado arriba del tiempo de falla
        Colht[k]=Lmes[str(HD+k)].values       #Extrae la demanda de la hora de falla y las horas siguientes segun la duración
    #==================================================================
    # MODIFICACIÓN DE LA CARGA DEL MODELO
    #==================================================================
    #print('Demanda de la hora' , k)
        net['load'].loc[:,'p_mw']=Colht.loc[:,k].values
    #==================================================================
    # CALCULAR EL FLUJO DE CARGA EN ESTADO NORMAL
    #==================================================================
        pp.runpp(net)
        RePerPLinN= net.res_line['p_from_mw']+net.res_line['p_to_mw']       # Guarda las perdidas P de las lineas en estado normal (a)
        RePerQLinN= net.res_line['q_from_mvar']+net.res_line['q_to_mvar']   # Guarda las perdidas Q de las lineas en estado normal (a)
    #==================================================================
    #CALCULO DE ENERGIA SUMINISTRADA EN ESTADO NORMAL
    #==================================================================
    #Calcular Energia suministarda al circuito despues de cada falla en cada una de las horas y la porción//Continua**
        if k <2:
            ESEN.iloc[k][Zona.iloc[0]['Proteccion']]=net.res_load['p_mw'].sum() #Extrae la suma de la energia total en las dos primeras horas
        else:
            ensn=net.res_load['p_mw'].sum() # Suma de la potencia en la hora 3
            tdc=ma.modf(TF)                 # Factor de multiplicación de la porción de la hora 3
            ESEN.iloc[k][Zona.iloc[0]['Proteccion']]=ensn*tdc[0]  #Agrega la fila al data frame
    #==================================================================
    # APERTURA DE ELEMENTO EN FALLA
    #==================================================================
        #print('Abrir Switch')
        inSw=pp.get_element_index(net,"switch", Zona.iloc[0]['Proteccion']) #Enceuntra el indice del elemento
        net['switch'].loc[inSw,'closed']=False  #Accede a al elemnto y lo abre
        #print(net['switch'].loc[inSw,'closed'])
    #==================================================================
    # EJECUCIÓN DEL FLUJO DE CARGA MODIFICADO
    #==================================================================
        SPPerL=0     #Definición de Variable almacenadora de perdidas P en las lineas (a)
        SQPerL=0     #Definición de Variable almacenadora de perdidas  Q en las lineas(a)
        pp.runpp(net)    #Ejecución de flujo de carga modificado
        Lip0=net.res_line[net.res_line['p_from_mw']==0]      #Filtrara lineas que tiene potencia cero (a)
        Liq0=net.res_line[net.res_line['q_from_mvar']==0]    #Filtrara lineas que tiene potencia cero (a)
        for z in range(len(Lip0)):
            SPPerL= SPPerL + RePerPLinN.loc[Lip0.index[z]]   # Acumular la potencia P de perdidas (a)
            SQPerL= SQPerL + RePerQLinN.loc[Liq0.index[z]]   # Acumular la potencia Q de perdidas (a)
    #==================================================================
    #CALCULO DE ENERGIA Y LA POTENCIA SUMINISTRADA EN FALLA
    #==================================================================
    #Calcular Energia suministarda al circuito despues de cada falla en cada una de las horas y la porciaón//Continua**
        if k <2:
            ESF.iloc[k][Zona.iloc[0]['Proteccion']]=net.res_load['p_mw'].sum() #Extrae la suma de la energia total en las dos primeras horas
            PSF.iloc[k][Zona.iloc[0]['Proteccion']]=net.res_load['p_mw'].sum() #Extrae la suma de la Potencia P total en las dos primeras horas
            QSF.iloc[k][Zona.iloc[0]['Proteccion']]=net.res_load['q_mvar'].sum() #Extrae la suma de la Potencia Q total en las dos primeras horas
            LPSF.iloc[k][Zona.iloc[0]['Proteccion']]=SPPerL                      #Extrae la suma de Potencia P de perdiadas de lineas no alimentadas en falla taridas  del estado normal de operación en las dos primeras horas (a)
            LQSF.iloc[k][Zona.iloc[0]['Proteccion']]=SQPerL                      #Extrae la suma de Potencia Q de perdiadas de lineas no alimentadas en falla taridas  del estado normal de operación en las dos primeras horas (a)
        else:
            enst=net.res_load['p_mw'].sum() # Suma de la potencia P en la hora 3
            qnst=net.res_load['q_mvar'].sum() # Suma de la potencia Q en la hora 3
            tdc=ma.modf(TF)                 # Factor de multiplicación de la porción de la hora 3
            ESF.iloc[k][Zona.iloc[0]['Proteccion']]=enst*tdc[0]  #Agrega la fila al data frame
            PSF.iloc[k][Zona.iloc[0]['Proteccion']]=enst  #Agrega la fila al data frame
            QSF.iloc[k][Zona.iloc[0]['Proteccion']]=qnst  #Agrega la fila al data frame
            LPSF.iloc[k][Zona.iloc[0]['Proteccion']]=SPPerL      #Agrega la fila al data frame de Potencia de perdidas P no suministarda (a)
            LQSF.iloc[k][Zona.iloc[0]['Proteccion']]=SQPerL      #Agrega la fila al data frame de Potencia de perdidas Q no suministarda (a)
    #======================================================================
    #CONTAR NUMERO DE CLEINTES DESATENDIDOS
    #======================================================================
        if k==0:
            Loce=net.res_load[net.res_load['p_mw']==0]   #Filtra las cargas que tienen cero en el flujo de carga
            coni=Loce.index.tolist() #Crea una lista con los indices a las correpoendientes  las cargas de cero
            if len(coni)==0:
                pass
            else:
                for l in range(len(coni)):
                    clien=Dload.iloc[coni]['Client']
                    SA1.iloc[coni[l]][Zona.iloc[0]['Proteccion']]=clien[coni[l]]
            Tclie=SA1.sum()
            #LOAD0[Zona.iloc[j]['Proteccion']]=coni
    #==========================================================================
    #CALCULO DE ENERGIA NO SUMIISTRADA EN FALLA
    #==========================================================================
    #Continiua**//Suma Energia total
    DIES=ESEN.sum()  #Suma la energia  de las cargas conectadas en estado normal
    ESEF=ESF.sum()   #Suma la energia  de las cargas conectadas durante el corte de energia
    DIF.loc[1]=DIES
    DIF.loc[2]=-ESEF
    ENS=DIF.sum()    #Diferencia de energia en estado normal con Energia estado en Falla de lo que resulta la ENS
    #==========================================================================
    #SUMA DE POTENCIA P y Q DE CARGAS Y PERDIDAS EN LAS LINEAS SIN CARGA EN FALLA EN CADA HORA
    #==========================================================================
    PTSF=PSF+LPSF
    QTSF=QSF+LQSF
    #==========================================================================
    # CIERRE DEL ELEMENTO EN FALLA
    #==========================================================================
    #print('Cerrar Switch')
    net['switch'].loc[inSw,'closed']=True  #Accede a al elemnto y lo cierra
    #print(net['switch'].loc[inSw,'closed'])
    #print('Cambio de Protección ')
    #print('==============================================================')
    return Tclie, ENS, PTSF, QTSF, ESF

#FUNCIÓN -SelecSwitch- SELECCIONA UN SWITCH ALEATORIAMENTE BASADO EN UNA DISTRIBUCIÓN UNIFORME EN RELACIÓN AL BUS DE PRUEBA (Bus es función del Switch)
def SelecSwitch (MatRe,BusA):
    print('================================================================================')
    print('Inicia Selección de Switch relacionado de forma Aleatorao en función de la Barra')
    print('================================================================================')
    print('Inicia')
    print('Procesando....')
    inBus=pp.get_element_index(net,"bus", BusA)     #Enceuntra el indice del elemento
    BFs=MatRe.loc[inBus,:]                          #Extrae la fila boleana del bus especifico para todo los Switchs
    BFsi=BFs.reset_index()                          #Agrega una columna de indices numerados nueva
    VecSw=BFsi.loc[BFsi.loc[:,inBus]==True]         #Filtra los valores Boleanos True, equivalentes a voltaje cero cuando se abre el Switch
    VecSwi=VecSw.reset_index()                      #Agrega una columna de indices numerados nueva
    IdxSw=int(np.random.uniform(0,len(VecSwi),1))   #Indice de la fila alaeatoria del vector de switches relacionados al Bus
    SwALE=VecSwi.loc[IdxSw,'name']
    print('Finaliza')                 #Extrae el nombre  del Switch escogido Aleatoreamente.
    return(SwALE, VecSw)

#FUNCIÓN -ModZonsinSwP- MODIFICA LAS ZONAS QUE NO TIENEN EL SWITCH DE PRUEBA, HALLANDO LA RELACIÓN ENTRE LOS SWICHES RELACIONADOS SL SW DE PRUEBA CON CADA ZONA QUE NO TIENEN EL SWITCH DE PRUEBA
def ModZonsinSwP (Zon,VecSwP):
    VecSwPi=VecSwP.reset_index()
    ViSz=np.zeros(shape=len(Zon))  #Crea un vector vacio de longtud del DF de la Zona1 con los indices de los swtchs de Zona3
    VrelB=np.zeros(shape=len(Zon))  #Crea un vector vacio de longtud del DF de la Zona1 para las relaciones de los swtches de Zona2
    for i in range(len(Zon)):
        iSz=pp.get_element_index(net,'switch',Zon.loc[i,'Proteccion']) #Halla el indice de Switch de Zona
        ViSz[i]=iSz #Remplza el vectro ceros con los indices de cda Switch de la RED
        for j in range(len(VecSwP)):
            iSr=pp.get_element_index(net,'switch',VecSwPi.loc[j,'name']) #Halla el indice de Switch de vector de swiches relacionados al bus de prueba
            if iSr==iSz:
                VrelB[i]=True #Agrega 'True' al vector de ceros si el hay un switch que afecte al bus de prueba, diferente al Switch de prueba
    Zon['idxR']=ViSz  #Agrega una nueva columna con los indices de la red de cada Switch
    Zon['BprS']=VrelB  #Agrega una nueva columna boleana con 'True', si hay swiches que afecten la barra de prueba diferente al switch de prueba
    ZonsinR=Zon.loc[Zon.loc[:,'BprS']==0]     #Switches sin relación al bus de prueba diferentes al switch de prueba
    ZonconR=Zon.loc[Zon.loc[:,'BprS']==True]  #Switches con relación al bus de prueba diferentes al switch de prueba
    ZonN=pd.concat([ZonsinR,ZonconR]) #Concatenar loas dataframes con y sin relación
    ZonF=ZonN.drop(['index'], axis=1)
    ZonF.reset_index() #Reiniciciar indice
    return(ZonF)

#FUNCIÓN -idenSwFall- DETERMINA SI EL SWITCH DE PRUEBA SE ENCUENTRA EN LA MUESTRA DE LAS FALLAS, DE LO CONTRARIO LO ASIGNA A LA ZONA A LA QUE PERTENECE (El switch remplaza a alguno de los simuldos en la muestra asi se mantien los parametros de tiempo de la muestra)
def idenSwFall (EstiP,Zona1,Zona2,Zona3,SWP,SwRB):
    Zon1ide=Zona1
    Zon2ide=Zona2
    Zon3ide=Zona3
    print('Zona1')
    print(Zon1ide)
    print('Zona2')
    print(Zon2ide)
    print('Zona3')
    print(Zon3ide)
    print('============================================================================**')
    print('Inicia validación de muestras de falla y modificacaión de ser necesario')
    print('============================================================================')
    print('')
    IdenZon=EstiP[EstiP['Proteccion']==SWP]   #Identificar zona del Switch (Filtra por nombre de Switches en la lista de Switches)
    IdenZon.index=[1]  #Renombrar el indice del dataframe
    iSp=pp.get_element_index(net,'switch',SWP) #Halla el indice de Switch de Prueba
    print('Indice del elemento de prueba en la Red: ', iSp)
    print('')
    Bz1=1   #Creación de Bandera Zona 1
    Bz2=1   #Creación de Bandera Zona 2
    Bz3=1   #Creación de Bandera Zona 3
    ISwr=0  #Inicializa indice aleatorio
    SwRBi=SwRB.reset_index()  #Reiniciar indices del vector

    if IdenZon.loc[1,'Zona']=='Z1':
        print('El Elemento esta en la  Zona1')
        ViSz=np.zeros(shape=len(Zon1ide))  #Crea un vector vacio de longtud del DF de la Zona1 con los indices de los swtches de Zona1
        VrelB=np.zeros(shape=len(Zon1ide))  #Crea un vector vacio de longtud del DF de la Zona1 para las relaciones de los swtches de Zona1
        for i in range(len(Zon1ide)):
            iSz=pp.get_element_index(net,'switch',Zon1ide.loc[i,'Proteccion']) #Halla el indice de Switch de Zona
            ViSz[i]=iSz #Remplaza el vector ceros con los indices de cada Switch de la RED
            if iSp==iSz: #Asigna cero si el switch esta en la muestra
                Bz1=0
                print('El Elemento esta en la Muestra de la Zona1')
            for j in range(len(SwRB)):
                iSr=pp.get_element_index(net,'switch',SwRBi.loc[j,'name']) #Halla el indice de Switch de vector de swiches relacionados al bus de prueba
                if iSr==iSz:
                    VrelB[i]=1 #Agrega 'True' al vector de ceros si  hay un switch que afecte al bus de prueba, diferente al Switch de prueba

        Zon1ide['idxR']=ViSz  #Agrega una nueva columna con los indices de la red de cada Switch
        Zon1ide['BprS']=VrelB  #Agrega una nueva columna boleana con 'True', si hay swiches que afecten la barra de prueba diferente al switch de prueba
        ZonasinR1=Zon1ide.loc[Zon1ide.loc[:,'BprS']==0]     #Switches sin relación al bus de prueba diferentes al switch de prueba
        ZonaconR1=Zon1ide.loc[Zon1ide.loc[:,'BprS']==1]     #Switches con relación al bus de prueba diferentes al switch de prueba
        #ZonasinR1=ZonasinR1.reset_index()  #Reiniciar indices del Dtaframe
        #ZonaconR1=ZonaconR1.reset_index()  #Reiniciar indices del Dtaframe
        print('Zonas 1 Sin Relación')
        print(ZonasinR1)
        print('Zonas 1 Con Relación')
        print(ZonaconR1)

        Zona1N=pd.concat([ZonasinR1,ZonaconR1]) #Concatenar los dataframes con y sin relación

        if Bz1==1: #Si el Switch no esta en la muestra lo agregara aleatroeamente al dataframe sin relación basado en una distribución uniforme
            print('Se Modifica la Zona 1')
            int(np.random.uniform(0,len(ZonasinR1),1))#int(np.random.uniform(0,len(ZonasinR1),1))   #Indice de la fila alaeatoria de la zona a modificar
            #ISwr=0#!!!!!!!!!!!!!!!!!!!!!!!!!!!!
            Zona1N.loc[ISwr,'Proteccion']=SWP
            Zona1N.loc[ISwr,'idxR']=iSp
            Zona1N.loc[ISwr,'BprS']=1

        Zona1F=Zona1N.drop(['index'], axis=1) #Elimina una colomna de indices que vien del generador de fallas
        Zona1F=Zona1N.drop(['level_0'], axis=1) #Elimina una colomna de indices que vien del generador de fallas
        Zona1F.reset_index() #Reiniciciar indice
        print('Zonas Concatenada')
        print(Zona1F)
        Zona2F= ModZonsinSwP (Zon2ide,SwRB) #Crea las Zonas donde no está el Switch de prueba para identificar cuales protecciones se relacionan al bus de prueba en otras zonas
        print('')
        print(Zona2F)
        Zona3F= ModZonsinSwP (Zona3,SwRB) #Crea las Zonas donde no está el Switch de prueba para identificar cuales protecciones se relacionan al bus de prueba en otras zonas
        print('')
        print(Zona3F)

    elif IdenZon.loc[1,'Zona']=='Z2':
        #print('El Elemento esta en la  Zona2')
        ViSz=np.zeros(shape=len(Zon2ide))  #Crea un vector vacio de longtud del DF de la Zona1 con los indices de los swtchs de Zona2
        VrelB=np.zeros(shape=len(Zon2ide))  #Crea un vector vacio de longtud del DF de la Zona1 para las relaciones de los swtches de Zona2
        for i in range(len(Zon2ide)):
            iSz=pp.get_element_index(net,'switch',Zon2ide.loc[i,'Proteccion']) #Halla el indice de Switch de Zona
            ViSz[i]=iSz #Remplza el vectro ceros con los indices de cda Switch de la RED
            if iSp==iSz: #Asigna cero si el switch esta en la muestra
                Bz2=0
                #print('El Elemento esta en la Muestra de la Zona2')
            for j in range(len(SwRB)):
                iSr=pp.get_element_index(net,'switch',SwRBi.loc[j,'name']) #Halla el indice de Switch de vector de swiches relacionados al bus de prueba
                if iSr==iSz:
                    VrelB[i]=1 #Agrega 'True' al vector de ceros si el hay un switch que afecte al bus de prueba, diferente al Switch de prueba

        Zon2ide['idxR']=ViSz  #Agrega una nueva columna con los indices de la red de cada Switch
        Zon2ide['BprS']=VrelB  #Agrega una nueva columna boleana con 'True', si hay swiches que afecten la barra de prueba diferente al switch de prueba
        ZonasinR2=Zon2ide.loc[Zon2ide.loc[:,'BprS']==0]     #Switches sin relación al bus de prueba diferentes al switch de prueba
        ZonaconR2=Zon2ide.loc[Zon2ide.loc[:,'BprS']==1]     #Switches con relación al bus de prueba diferentes al switch de prueba
        #ZonasinR2=ZonasinR2.reset_index()  #Reiniciar indices del Dtaframe
        #ZonaconR2=ZonaconR2.reset_index()  #Reiniciar indices del Dtaframe
        print('Zonas 2 Sin Relación')
        print(ZonasinR2)
        print('Zonas 2 Con Relación')
        print(ZonaconR2)

        Zona2N=pd.concat([ZonasinR2,ZonaconR2]) #Concatenar loas dataframes con y sin relación

        if Bz2==1: #Si el Switch no esta en la muestra lo agregara aleatroeamente basado en una distribución uniforme
            print('Se Modifica la Zona 2')
            int(np.random.uniform(0,len(ZonasinR2),1))#int(np.random.uniform(0,len(ZonasinR2),1))   #Indice de la fila alaeatoria de la zona a modificar
            #ISwr=0#!!!!!!!!!!!!!!!!!!!!!!!!
            Zona2N.loc[ISwr,'Proteccion']=SWP
            Zona2N.loc[ISwr,'idxR']=iSp
            Zona2N.loc[ISwr,'BprS']=1

        Zona2F=Zona2N.drop(['index'], axis=1)
        Zona2F=Zona2N.drop(['level_0'], axis=1)
        Zona2F.reset_index() #Reiniciciar indice
        print('Zonas Concatenada')
        print(Zona2F)
        Zona1F= ModZonsinSwP (Zon1ide,SwRB) #Crea las Zonas donde no está el Switch de prueba para identificar cuales protecciones se relacionan al bus de prueba en otras zonas
        print('')
        print(Zona1F)
        Zona3F= ModZonsinSwP (Zon3ide,SwRB) #Crea las Zonas donde no está el Switch de prueba para identificar cuales protecciones se relacionan al bus de prueba en otras zonas
        print('')
        print(Zona3F)

    else:
        #print('El Elemento esta en la  Zona3')
        ViSz=np.zeros(shape=len(Zon3ide))  #Crea un vector vacio de longtud del DF de la Zona1 con los indices de los swtchs de Zona3
        VrelB=np.zeros(shape=len(Zon3ide))  #Crea un vector vacio de longtud del DF de la Zona1 para las relaciones de los swtches de Zona2
        for i in range(len(Zon3ide)):
            iSz=pp.get_element_index(net,'switch',Zon3ide.loc[i,'Proteccion']) #Halla el indice de Switch de Zona
            ViSz[i]=iSz #Remplza el vectro ceros con los indices de cda Switch de la RED
            if iSp==iSz: #Asigna cero si el switch esta en la muestra
                Bz3=0
                #print('El Elemento esta en la Muestra de la Zona3')
            for j in range(len(SwRB)):
                iSr=pp.get_element_index(net,'switch',SwRBi.loc[j,'name']) #Halla el indice de Switch de vector de swiches relacionados al bus de prueba
                if iSr==iSz:
                    VrelB[i]=1 #Agrega 'True' al vector de ceros si el hay un switch que afecte al bus de prueba, diferente al Switch de prueba

        Zon3ide['idxR']=ViSz  #Agrega una nueva columna con los indices de la red de cada Switch
        Zon3ide['BprS']=VrelB  #Agrega una nueva columna boleana con 'True', si hay swiches que afecten la barra de prueba diferente al switch de prueba
        ZonasinR3=Zon3ide.loc[Zon3ide.loc[:,'BprS']==0]     #Switches sin relación al bus de prueba diferentes al switch de prueba
        ZonaconR3=Zon3ide.loc[Zon3ide.loc[:,'BprS']==1]  #Switches con relación al bus de prueba diferentes al switch de prueba
        #ZonasinR3=ZonasinR3.reset_index()  #Reiniciar indices del Dtaframe
        #ZonaconR3=ZonaconR3.reset_index()  #Reiniciar indices del Dtaframe
        print('Zonas 3 Sin Relación')
        print(ZonasinR3)
        print('Zonas 3 Con Relación')
        print(ZonaconR3)

        Zona3N=pd.concat([ZonasinR3,ZonaconR3]) #Concatenar los dataframes con y sin relación

        if Bz3==1: #Si el Switch no esta en la muestra lo agregara aleatroeamente basado en una distribución uniforme
            print('Se Modifica la Zona 3')
            int(np.random.uniform(0,len(ViSz),1))#int(np.random.uniform(0,len(ViSz),1))   #Indice de la fila alaeatoria de la zona a modificar
            #ISwr=0# !!!!!!!!!!!!!!!!!!
            Zona3N.loc[ISwr,'Proteccion']=SWP
            Zona3N.loc[ISwr,'idxR']=iSp
            Zona3N.loc[ISwr,'BprS']=1


        Zona3F=Zona3N.drop(['index'], axis=1)
        Zona3F=Zona3N.drop(['level_0'], axis=1)
        Zona3F=Zona3F.reset_index() #Reiniciciar indice
        print('Zonas Concatenada')
        print(Zona3F)
        Zona1F= ModZonsinSwP (Zon1ide,SwRB) #Crea las Zonas donde no está el Switch de prueba para identificar cuales protecciones se relacionan al bus de prueba en otras zonas
        print('Zona F1')
        print(Zona1F)
        Zona2F= ModZonsinSwP (Zon2ide,SwRB) #Crea las Zonas donde no está el Switch de prueba para identificar cuales protecciones se relacionan al bus de prueba en otras zonas
        print('Zona F2')
        print(Zona2F)

    print('')
    print('Finaliza')
    return(Zona1F, Zona2F, Zona3F)

#FUNCIÓN-EPQclient_ZA- DETERMINA LA ENERGIA NO SUMINISTRADA Y SUMINISTRADA, EL NUMERO DE CLIENTES, POTENCIA ACTIVA Y REACTIVA DURANTE LA FALLA EN UNA ZONA
def EPQclient_ZA (Zona):
    print(Zona)
    lt=ma.ceil(max(Zona.loc[:,'tiempo_pr_fall'])) #Redondea hacia arriba el tiempo maximo de falla en la muestra
    PSF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))        #Dataframe para guardar la potencia p en falla para cada protección
    QSF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))        #Dataframe para guardar la potencia Q en falla para cada protección
    LPSF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))       #Dataframe para guardar la potencia de perdidas p en falla para cada protección (a)
    LQSF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))       #Dataframe para guardar la potencia de perdidas Q en falla para cada protección (a)
    ESF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))        #Dataframe para guardar la energia en falla
    ESEN=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))       #Dataframe para guardar la energia en estado normal
    PSEN=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))       #Dataframe para guardar la potencia p en estado normal para cada protección
    QSEN=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))      #Dataframe para guardar la potencia Q en estado normal para cada protección
    DIES=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))       #Diferencia de la energia
    SA1=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(len(Dload)))  #Dataframe para almacenar los clientes sin servicio en cada falla
    DIF=pd.DataFrame(columns=Zona.loc[:,'Proteccion'],index=range(0,lt))        #Diferencia de la energia
    for j in range(len(Zona)):
        DA=Zona.iloc[j]['Dia_dl_Ano']          #Extrae el día de la falla
        HD=int(Zona.iloc[j]['Hora_dl_Dia'])    #Extrae la hora de la falla
        TF=Zona.iloc[j]['tiempo_pr_fall']      #Extrae el tiempo de la falla
        Colht = pd.DataFrame()                 #Crea un DF temporal vacio
        Lmes=SelMesDeman(Labr, Lmay, Ljun, Ljul, Lago, Lsep, Loct, DA) #Señecciona el mes de la demanda baaso en el dia de falla
        #print("---------------------------------------")
        #print(Zona.iloc[j]['Proteccion'])
        #print('->Falla ocurrida el día ' + str(DA))
        #print('->En la hora ' + str(HD))
        #print('->Durante ' + str(TF)+'Horas')
        #print("---------------------------------------")
        clien=[]
        for k in range(ma.ceil(TF)):              #Repite dependiendo de el numero redondeado arriba del tiempo de falla
            Colht[k]=Lmes[str(HD+k)].values       #Extrae la demanda de la hora de falla y las horas siguientes segun la duración
            #==================================================================
            # MODIFICACIÓN DE LA CARGA DEL MODELO
            #==================================================================
            #print('Demanda de la hora' , k)
            net['load'].loc[:,'p_mw']=Colht.loc[:,k].values
            #==================================================================
            # CALCULAR EL FLUJO DE CARGA EN ESTADO NORMAL
            #==================================================================
            pp.runpp(net)
            RePerPLinN= net.res_line['p_from_mw']+net.res_line['p_to_mw']       # Guarda las perdidas P de las lineas en estado normal (a)
            RePerQLinN= net.res_line['q_from_mvar']+net.res_line['q_to_mvar']   # Guarda las perdidas Q de las lineas en estado normal (a)
            #==================================================================
            #CALCULO DE ENERGIA SUMINISTRADA EN ESTADO NORMAL
            #==================================================================
            #Calcular Energia suministarda al circuito despues de cada falla en cada una de las horas y la porción//Continua**
            if k <lt-1:
                ESEN.iloc[k][Zona.iloc[j]['Proteccion']]=net.res_load['p_mw'].sum() #Extrae la suma de la energia total en las dos primeras horas En etdo Normal
                print('CALCULO ESTADO NORMAL')
                #print(ESEN)
                PSEN.iloc[k][Zona.iloc[j]['Proteccion']]=net.res_load['p_mw'].sum() #Extrae la suma de la Potencia P total en las dos primeras horas En estado Normal
                QSEN.iloc[k][Zona.iloc[j]['Proteccion']]=net.res_load['q_mvar'].sum() #Extrae la suma de la Potencia Q total en las dos primeras horas En Estado Normal
            else:
                esn=net.res_load['p_mw'].sum() # Suma de la potencia en la hora 3
                qst=net.res_load['q_mvar'].sum() # Suma de la potencia Q en la hora 3
                tdc=ma.modf(TF)                 # Factor de multiplicación de la porción de la hora 3
                ESEN.iloc[k][Zona.iloc[j]['Proteccion']]=esn*tdc[0]  #Agrega la fila al data frame
                print('CALCULO ESTADO NORMAL')
                #print(ESEN)
                PSEN.iloc[k][Zona.iloc[j]['Proteccion']]=esn  #Agrega la fila al data frame
                QSEN.iloc[k][Zona.iloc[j]['Proteccion']]=qst  #Agrega la fila al data frame
            #==================================================================
            # APERTURA DE ELEMENTO EN FALLA
            #==================================================================
            #print('Abrir Switch')
            inSw=pp.get_element_index(net,"switch", Zona.iloc[j]['Proteccion']) #Enceuntra el indice del elemento
            net['switch'].loc[inSw,'closed']=False  #Accede a al elemento y lo abre
            #print(net['switch'].loc[inSw,'closed'])
            #==================================================================
            # EJECUCIÓN DEL FLUJO DE CARGA MODIFICADO
            #==================================================================
            SPPerL=0     #Definición de Variable almacenadora de perdidas P en las lineas (a)
            SQPerL=0     #Definición de Variable almacenadora de perdidas  Q en las lineas(a)
            pp.runpp(net)    #Ejecución de flujo de carga modificado
            Lip0=net.res_line[net.res_line['p_from_mw']==0]      #Filtrara lineas que tiene potencia cero (a)
            Liq0=net.res_line[net.res_line['q_from_mvar']==0]    #Filtrara lineas que tiene potencia cero (a)
            for z in range(len(Lip0)):
                SPPerL= SPPerL + RePerPLinN.loc[Lip0.index[z]]   # Acumular la potencia P de perdidas (a)
                SQPerL= SQPerL + RePerQLinN.loc[Liq0.index[z]]   # Acumular la potencia Q de perdidas (a)
            #==================================================================
            #CALCULO DE ENERGIA Y LA POTENCIA SUMINISTRADA EN FALLA
            #==================================================================
            #Calcular Energia suministarda al circuito despues de cada falla en cada una de las horas y la porción//Continua**
            if k <lt-1:
                ESF.iloc[k][Zona.iloc[j]['Proteccion']]=net.res_load['p_mw'].sum()   #Extrae la suma de la energia total en las dos primeras horas
                print('CALCULO ESTADO FALLA')
                #print(ESF)
                PSF.iloc[k][Zona.iloc[j]['Proteccion']]=net.res_load['p_mw'].sum()   #Extrae la suma de la Potencia P total en las dos primeras horas
                QSF.iloc[k][Zona.iloc[j]['Proteccion']]=net.res_load['q_mvar'].sum() #Extrae la suma de la Potencia Q total en las dos primeras horas
                LPSF.iloc[k][Zona.iloc[j]['Proteccion']]=SPPerL                      #Extrae la suma de Potencia P de perdiadas de lineas no alimentadas en falla taridas  del estado normal de operación en las dos primeras horas (a)
                LQSF.iloc[k][Zona.iloc[j]['Proteccion']]=SQPerL                      #Extrae la suma de Potencia Q de perdiadas de lineas no alimentadas en falla taridas  del estado normal de operación en las dos primeras horas (a)
            else:
                enst=net.res_load['p_mw'].sum()   # Suma de la potencia P en la hora 3
                qnst=net.res_load['q_mvar'].sum() # Suma de la potencia Q en la hora 3
                tdc=ma.modf(TF)                   # Factor de multiplicación de la porción de la hora 3
                ESF.iloc[k][Zona.iloc[j]['Proteccion']]=enst*tdc[0]  #Agrega la fila al data frame
                print('CALCULO ESTADO FALLA')
                #print(ESF)
                PSF.iloc[k][Zona.iloc[j]['Proteccion']]=enst  #Agrega la fila al data frame
                QSF.iloc[k][Zona.iloc[j]['Proteccion']]=qnst  #Agrega la fila al data frame
                LPSF.iloc[k][Zona.iloc[j]['Proteccion']]=SPPerL      #Agrega la fila al data frame de Potencia de perdidas P no suministarda (a)
                LQSF.iloc[k][Zona.iloc[j]['Proteccion']]=SQPerL      #Agrega la fila al data frame de Potencia de perdidas Q no suministarda (a)
            #======================================================================
            #CONTAR NUMERO DE CLEINTES DESATENDIDOS
            #======================================================================
            if k==0:
                Loce=net.res_load[net.res_load['p_mw']==0]   #Filtra las cargas que tienen cero en el flujo de carga
                coni=Loce.index.tolist()                     #Crea una lista con los indices a las correpoendientes  las cargas de cero
                if len(coni)==0:
                    pass
                else:
                    for l in range(len(coni)):
                        clien=Dload.iloc[coni]['Client']
                        SA1.iloc[coni[l]][Zona.iloc[j]['Proteccion']]=clien[coni[l]]
                Tclie=SA1.sum() #Numero de clientes desatendidos
                #LOAD0[Zona.iloc[j]['Proteccion']]=coni
            #==========================================================================
            # CIERRE DEL ELEMENTO EN FALLA
            #==========================================================================
            print('Cerrar Switch')
            net['switch'].loc[inSw,'closed']=True  #Accede a al elemnto y lo cierra
            #print(net['switch'].loc[inSw,'closed'])
            print('Cambio de hora')
        #==========================================================================
        #CALCULO DE ENERGIA NO SUMIISTRADA EN FALLA
        #==========================================================================
        DIES=ESEN.sum()  #Suma la energia  de las cargas conectadas en estado normal
        ESEF=ESF.sum()   #Suma la energia  de las cargas conectadas durante el corte de energia
        DIF.loc[1]=DIES
        DIF.loc[2]=-ESEF
        ENS=DIF.sum()    #Diferencia de energia en estado normal con Energia estado en Falla de lo que resulta la ENS
        #==========================================================================
        #SUMA DE POTENCIA P y Q DE CARGAS Y PERDIDAS EN LAS LINEAS SIN CARGA EN FALLA EN CADA HORA
        #==========================================================================
        PTSF=PSF+LPSF
        QTSF=QSF+LQSF
        PISL=PSEN-PTSF
        QISL=QSEN-QTSF
        #==========================================================================
        # CIERRE DEL ELEMENTO EN FALLA
        #==========================================================================
        #print('Cerrar Switch Seguridad')
        net['switch'].loc[inSw,'closed']=True  #Accede a al elemnto y lo cierra
        #print(net['switch'].loc[inSw,'closed'])
        #print('Cambio de Protección')
    return Tclie, ENS, PISL, QISL #ESEN, ESF, PTSF, PSEN,

#FUNCIÓN -CalPBess- CALCULA EL ENS LA POTENCIA P Y Q EN CADA ZONA PARA LOS SWTCHS RELACIONADOS AL BUS DE FALLA
def CalPBess (Zona1r,Zona2r,Zona3r):
    PBess=pd.DataFrame(columns=['Zona1','Zona2','Zona3'],index=range(0,3))
    Zon1A=Zona1r.loc[Zona1r.loc[:,'BprS']==1] #Filtra Zona1, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    Zon2A=Zona2r.loc[Zona2r.loc[:,'BprS']==1] #Filtra Zona2, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    Zon3A=Zona3r.loc[Zona3r.loc[:,'BprS']==1] #Filtra Zona3, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    #restablecer el índice
    Zon1A.reset_index(drop=True, inplace=True)
    Zon2A.reset_index(drop=True, inplace=True)
    Zon3A.reset_index(drop=True, inplace=True)
    LZ1A=len(Zon1A) #Calcula la longitud del DataFrame de la Zona1
    LZ2A=len(Zon2A) #Calcula la longitud del DataFrame de la Zona2
    LZ3A=len(Zon3A) #Calcula la longitud del DataFrame de la Zona3
    #print(Zon1A)
    #print(Zon2A)
    #print(Zon3A)

    if LZ1A!=0:
        EPQCz1=EPQclient_ZA (Zon1A)  #Calculo  el numero de clientes no atendidos, la ENS, PISL, QISL
        ENSBesZ1= EPQCz1[1]
        MxENSBesZ1=ENSBesZ1.max().max()
        PBessZ1= EPQCz1[2]
        MxPBesZ1=PBessZ1.max().max()
        QBesZ1= EPQCz1[3]
        MxQBesZ1=QBesZ1.min().min()
    else:
        MxENSBesZ1=0
        MxPBesZ1=0
        MxQBesZ1=0
    if LZ2A!=0:
        EPQCz2=EPQclient_ZA (Zon2A)  #Calculo  el numero de clientes no atendidos, la ENS, PISL, QISL
        ENSBesZ2= EPQCz2[1]
        MxENSBesZ2=ENSBesZ2.max().max()
        PBessZ2= EPQCz2[2]
        MxPBesZ2=PBessZ2.max().max()
        QBesZ2= EPQCz2[3]
        MxQBesZ2=QBesZ2.min().min()
    else:
        MxENSBesZ2=0
        MxPBesZ2=0
        MxQBesZ2=0
    if LZ3A!=0:
        EPQCz3=EPQclient_ZA (Zon3A)  #Calculo  el numero de clientes no atendidos, la ENS, PISL, QISL
        ENSBesZ3= EPQCz3[1]
        MxENSBesZ3=ENSBesZ3.max().max()
        PBessZ3= EPQCz3[2]
        MxPBesZ3=PBessZ3.max().max()
        QBesZ3= EPQCz3[3]
        MxQBesZ3=QBesZ3.min().min()
    else:
        MxENSBesZ3=0
        MxPBesZ3=0
        MxQBesZ3=0
    PBess.iloc[0]=[MxENSBesZ1,MxENSBesZ2,MxENSBesZ3]  #Agrega la fila de la Energia E maxima en cada zona
    PBess.iloc[1]=[MxPBesZ1,MxPBesZ2,MxPBesZ3]        #Agrega la fila de la Potencia P maxima en cada zona (Columnas)
    PBess.iloc[2]=[MxQBesZ1,MxQBesZ2,MxQBesZ3]        #Agrega la fila de la Potencia Q maxima en cada zona (Columnas)
    PaPBess=PBess.max(axis=1)  #Parametros del BESS Fiala1:Energia, Fila2:PotenciaP, Fila3:PotenciaQ (Halla el maximo por filas del Dataframe)
    #print(PaPBess)
    return(PaPBess)

#FUNCIÓN -Calindx_CTOcb- CALCULA LOS INDICES DE CALIDAD SAIDI, DAIFI Y ENS EN EL CIRCUITO CON BEES
def Calindx_CTOcb (Zona1, Zona2, Zona3):
    print('*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-')
    print('Inicia Calculo de Indices de calidad CON BESS')
    print('*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*-')
    Zon1Bcb=Zona1.loc[Zona1.loc[:,'BprS']==0] #Filtra Zona1, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    Zon2Bcb=Zona2.loc[Zona2.loc[:,'BprS']==0] #Filtra Zona2, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    Zon3Bcb=Zona3.loc[Zona3.loc[:,'BprS']==0] #Filtra Zona3, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    print('Zona 1--------------------------------------------->')
    print(Zon1Bcb)
    print(len(Zon1Bcb))
    print('Zona 2--------------------------------------------->')
    print(Zon2Bcb)
    print(len(Zon2Bcb))
    print('Zona 3--------------------------------------------->')
    print(Zon3Bcb)
    print(len(Zon3Bcb))
    Bti1=0 #Bandera para identificar si la zona esta vacia  1:Vacio, 0:Completa
    Bti2=0 #Bandera para identificar si la zona esta vacia  1:Vacio, 0:Completa
    Bti3=0 #Bandera para identificar si la zona esta vacia  1:Vacio, 0:Completa
    #Condicional de calculo en caso de que alguna zona notenga ningun Switch realcionado, asigna ceros
    if len(Zon1Bcb)!=0:
        CEPQz1=EPQclient_ZA (Zon1Bcb)
    else:
        CEPQz1=np.array([np.array([0]) ,np.array([0]) ,np.array([0]) ,np.array([0]) ])
        Bti1=1
    if len(Zon2Bcb)!=0:
        CEPQz2=EPQclient_ZA (Zon2Bcb)
    else:
        CEPQz2=np.array([np.array([0]) ,np.array([0]) ,np.array([0]) ,np.array([0]) ])
        Bti2=1
    if len(Zon3Bcb)!=0:
        CEPQz3=EPQclient_ZA (Zon3Bcb)
    else:
        CEPQz3=np.array([np.array([0]) ,np.array([0]) ,np.array([0]) ,np.array([0]) ])
        Bti3=1
    #==============================================================================
    # PREPARACIÓN DE DATOS PARA CALCULO DE SAIDI, SAIFI y ENS
    #==============================================================================
    #Zona 1
    Ui1=np.array(CEPQz1[0])                            #Creación de array de numero de clientes sin servicio en Z1
    Ei1=np.array(CEPQz1[1])                            #Creación de array de ENS sin servicio en Z1
    if Bti1==0: #Condicional de zona Vacia
        ti1=np.array(Zon1Bcb.loc[:,'tiempo_pr_fall'])  #Creación de array de tiempo sin servicio en Z1
    else:
        ti1=np.array([0])
    #Zona 2
    Ui2=np.array(CEPQz2[0])                            #Creación de array de numero de clientes sin servicio en Z2
    Ei2=np.array(CEPQz2[1])                            #Creación de array de ENS sin servicio en Z2
    if Bti2==0: #Condicional de zona Vacia
        ti2=np.array(Zon2Bcb.loc[:,'tiempo_pr_fall'])  #Creación de array de tiempo sin servicio en Z2
    else:
        ti2=np.array([0])
    #Zona 3
    Ui3=np.array(CEPQz3[0])                            #Creación de array de numero de clientes sin servicio en Z3
    Ei3=np.array(CEPQz3[1])                            #Creación de array de ENS sin servicio en Z1
    if Bti3==0: #Condicional de zona Vacia
        ti3=np.array(Zon3Bcb.loc[:,'tiempo_pr_fall'])  #Creación de array de tiempo sin servicio en Z3
    else:
        ti3=np.array([0])

    print('Zona 1---------------------------------------------**************>')
    print(Ui1)
    print(Ei1)
    print(ti1)
    print('Zona 2---------------------------------------------**************>')
    print(Ui2)
    print(Ei2)
    print(ti2)
    print('Zona 3---------------------------------------------**************>')
    print(Ui3)
    print(Ei3)
    print(ti3)
    Ui=np.concatenate((Ui1,Ui2,Ui3))               #Union de arrays de clientes de las tres zonas
    ti=np.concatenate((ti1,ti2,ti3))               #Union de arrays de tiempo de las tres zonas
    ENSi=np.concatenate((Ei1,Ei2,Ei3))             #Union de arrays de energia en las tres zonas
    #==============================================================================
    # CALCULO DE SAIDI
    #==============================================================================
    SumSup=np.sum(Ui*ti)                         #Sumaproducto de Ui y ti
    SAIDIO=SumSup/Dload.loc[:,'Client'].sum()     #Divición en el numero de clientes, Optención SAIDI sin intervención BESS
    #print(SAIDIO)
    #==============================================================================
    # CALCULO DE SAIFI
    #==============================================================================
    SAIFIO=np.sum(Ui)/Dload.loc[:,'Client'].sum() #suma de clintes no servidos dividido en el numero de clientes, Optención SAIFI sin intervención BESS
    #print(SAIFIO)
    #==============================================================================
    # CALCULO DE LA ENERGIA NO SUMINISTRADA
    #==============================================================================
    ENSfO=np.sum(ENSi)                            #Suma ENS en cada una de las zonas sin intervención BESS
    #print(ENSfO)
    #==============================================================================
    return(SAIDIO,SAIFIO,ENSfO)

#FUNCIÓN -Calindx_CTOsb- CALCULA LOS INDICES DE CALIDAD SAIDI, DAIFI Y ENS EN EL CIRCUITO SIN BEES
def Calindx_CTOsb (Zona1, Zona2, Zona3):
    #Zon1B=Zona1.loc[Zona1.loc[:,'BprS']==0] #Filtra Zona1, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    #Zon2B=Zona2.loc[Zona2.loc[:,'BprS']==0] #Filtra Zona2, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    #Zon3B=Zona3.loc[Zona3.loc[:,'BprS']==0] #Filtra Zona3, si tiene relación con el Switch de Falla dejando solo los Switchs relacionados
    print(Zona1)
    print(Zona2)
    print(Zona3)
    print('*********************************************')
    print('Inicia Calculo de Indices de calidad SIN BESS')
    print('*********************************************')
    CEPQz1sb=EPQclient_ZA (Zona1)
    CEPQz2sb=EPQclient_ZA (Zona2)
    CEPQz3sb=EPQclient_ZA (Zona3)
    #==============================================================================
    # PREPARACIÓN DE DATOS PARA CALCULO DE SAIDI, SAIFI y ENS
    #==============================================================================
    Ui1=np.array(CEPQz1sb[0])                        #Creación de array de numero de clientes sin servicio en Z1
    Ei1=np.array(CEPQz1sb[1])                        #Creación de array de ENS sin servicio en Z1
    ti1=np.array(Zona1.loc[:,'tiempo_pr_fall']) #Creación de array de tiempo sin servicio en Z1
    Ui2=np.array(CEPQz2sb[0])                        #Creación de array de numero de clientes sin servicio en Z2
    Ei2=np.array(CEPQz2sb[1])                        #Creación de array de ENS sin servicio en Z2
    ti2=np.array(Zona2.loc[:,'tiempo_pr_fall']) #Creación de array de tiempo sin servicio en Z2
    Ui3=np.array(CEPQz3sb[0])                        #Creación de array de numero de clientes sin servicio en Z3
    Ei3=np.array(CEPQz3sb[1])                        #Creación de array de ENS sin servicio en Z1
    ti3=np.array(Zona3.loc[:,'tiempo_pr_fall']) #Creación de array de tiempo sin servicio en Z3
    Ui=np.concatenate((Ui1,Ui2,Ui3))             #Union de arrays de clientes de las tres zonas
    ti=np.concatenate((ti1,ti2,ti3))             #Union de arrays de tiempo de las tres zonas
    ENSisb=np.concatenate((Ei1,Ei2,Ei3))           #Union de arrays de energia en las tres zonas
    #==============================================================================
    # CALCULO DE SAIDI
    #==============================================================================
    SumSup=np.sum(Ui*ti)                         #Sumaproducto de Ui y ti
    SAIDIsb=SumSup/Dload.loc[:,'Client'].sum()     #Divición en el numero de clientes, Optención SAIDI sin intervención BESS
    #print(SAIDIsb)
    #==============================================================================
    # CALCULO DE SAIFI
    #==============================================================================
    SAIFIsb=np.sum(Ui)/Dload.loc[:,'Client'].sum() #suma de clintes no servidos dividido en el numero de clientes, Optención SAIFI sin intervención BESS
    #print(SAIFIsb)
    #==============================================================================
    # CALCULO DE LA ENERGIA NO SUMINISTRADA
    #==============================================================================
    ENSfsb=np.sum(ENSisb)                            #Suma ENS en cada una de las zonas sin intervención BESS
    #print(ENSfsb)
    #==============================================================================
    return(SAIDIsb,SAIFIsb,ENSfsb)

#FUNCIÓN -ConsulIDX- CALCULA LOS INDICES DE CALIDAD SAIDI, DAIFI, ENS Y LA POTENCIA DEL BESS  EN EL CIRCUITO SIN BEES
def ConsulIDX (BusPPyoF):
    #Se selecciona aleatoriamente el Switcn de falla relacionado al Bus de Prueba
    SwitchP=SelecSwitch(BfunS,BusPPyoF) #Selección aleatoria del Switch basado en la relación con el bus seleccionado
    print('Este es mi Switch de prueba')
    print(SwitchP[0])
    print('Este es mi Switch Rela de prueba')
    print(SwitchP[1])
    print('Este es mi bus de prueba FIN')
    #Valida si el Switch de prueba se enceuntra en las muestras de falla, de lo contrario lo asigna remplazando el nombre de alguno de ellos aleatoriamente, Tambien identifica relaciones del bus en las zonas restantes
    ZonasP=idenSwFall(EstiP, Zona1t, Zona2t, Zona3t, SwitchP[0],SwitchP[1])  #Zonas de Prueba
    print(ZonasP)
    #Calcar Indices en CON intervención BESS
    IndxcB= Calindx_CTOcb(ZonasP[0],ZonasP[1],ZonasP[2])

    #Calcar Indices en SIN intervención BESS
    IndxsB= Calindx_CTOsb(ZonasP[0],ZonasP[1],ZonasP[2])

    #Identifica la potencia y energia que debera tener cada BEES durante la falla, en función del Switch relacionado al bus de prueba en cada zona
    ParamBess=CalPBess(ZonasP[0],ZonasP[1],ZonasP[2])
    #print(ParamBess)
    return(IndxcB,ParamBess,IndxsB)

#FUNCIÓN -SimCasEst- SIMULACIÓN DE CASOS DE ESTUDIO PARA HALLAR RESULTADOS EN CADA UNA DE LAS BARRAS
def SimCasEst (PAGeo,PACost,FCase,tipBess):
    PARAM_1=pd.DataFrame(columns=['BusN','SAIDI_Bi-1','SAIFI_Bi-1','ENE_Bi-1'],index=range(len(PAGeo))) #Parametros en estado i-1 (Normal sin Bess)
    PARAM=pd.DataFrame(columns=['BusN','FO','SAIDI_Bi','SAIFI_Bi','ENE_Bi','CT_Bi','RG_Bi'],index=range(len(PAGeo))) #Parametros en estado i con BESS
    PARAM_Bess=pd.DataFrame(columns=['BusN','PW','EW'],index=range(len(PAGeo))) #Parametros en estado i con BESS
    PARAM_1.loc[:,'BusN']=PAGeo.loc[:,'BUS']
    PARAM.loc[:,'BusN']=PAGeo.loc[:,'BUS']
    PARAM_Bess.loc[:,'BusN']=PAGeo.loc[:,'BUS']
    #------------------------------------------------------------------------------
    #Definir Modelo
    #------------------------------------------------------------------------------
    #for i in range(len(PAGeo)):
    for i in range(0,2):
        #--------------------------------------------------------------------------
        #Consultar Varables
        #--------------------------------------------------------------------------
        BusPyo=PAGeo.loc[i,'BUS']                 #Extrae el bus de la lista de parametros
        IndyPE= ConsulIDX (BusPyo)                #Calcula el SIDI, SAIFI, ENS PotenciaP y Energia del BESS
        INDICESi=IndyPE[0]                        #Asigana los indices calculados segun el bus escogido
        PE=IndyPE[1]                              #Calcula la potencia ye energia del BESS segun el bus escogido
        INDICESi_1=IndyPE[2]                      #Asigana los indices calculados segun el bus escogido
        #--------------------------------------------------------------------------
        #Calculo de indices de referencia i-1 (Normal sin Bess)
        #--------------------------------------------------------------------------
        PARAM_1.loc[i,'SAIDI_Bi-1']=INDICESi_1[0]     #Consulta y guarda el SAIDI calculado en estado i-1 (Normal sin Bess)
        PARAM_1.loc[i,'SAIFI_Bi-1']=INDICESi_1[1]     #Consulta y guarda el SAIFI calculado en estado i-1 (Normal sin Bess)
        PARAM_1.loc[i,'ENE_Bi-1']=INDICESi_1[2]       #Consulta y guarda el ENS calculado en estado i-1 (Normal sin Bess)
        #--------------------------------------------------------------------------
        #Establecer Indices
        #--------------------------------------------------------------------------
        a=FCase[0]   #Factor SAIDIP
        b=FCase[1]   #Factor SAIFI
        c=FCase[2]   #Factor ENS
        d=FCase[3]   #Factor COSTO
        e=FCase[4]   #Factor Elevación Geografica
        f=FCase[5]   #Factor Acsecibilidad
        g=FCase[6]   #Factor de Locación Urbano/Rural
        h=FCase[7]   #Factor de espacio
        l=FCase[8]   #Factor de Forestal
        #--------------------------------------------------------------------------
        #Consultar los indicesa calculados
        #--------------------------------------------------------------------------
        SAIDIi=float(INDICESi[0])                       #Consulta el SAIDI calculado
        SAIFIi=float(INDICESi[1])                       #Consulta el SAIFI calculado
        ENSi=float(INDICESi[2])                         #Consulta el ENS calculado
        EBessi=float(PE[0])                             #Consulta el Energia del Bess calculada
        PBessi=float(PE[1])                             #Consulta el Potencia del Bess calculada
        #--------------------------------------------------------------------------
        #Calculo de Parametro de Costo Asociado del Bess
        #--------------------------------------------------------------------------
        CTi=(PBessi*PACost.loc[tipBess,'Ppot'])+(EBessi*PACost.loc[tipBess,'Eenergy'])+(PBessi*PACost.loc[tipBess,'OM'])
        #--------------------------------------------------------------------------
        #Función Objetivo
        #--------------------------------------------------------------------------
        FOi=a*SAIDIi+b*SAIFIi+c*ENSi+d*CTi
        #--------------------------------------------------------------------------
        #Calculo de Parametro de Riengo Geografico
        #--------------------------------------------------------------------------
        RGi=(e*PAGeo.loc[i,'NorC_Eleva'])+(f*PAGeo.loc[i,'NorC_Via'])+(g*PAGeo.loc[i,'NorC_Localiza'])+(h*PAGeo.loc[i,'NorC_Espacio'])+(l*PAGeo.loc[i,'NorC_Forestal'])
        #--------------------------------------------------------------------------
        #Almacenamiento de Variables, FO y Parametros
        #--------------------------------------------------------------------------
        PARAM.loc[i,'FO']=FOi                  #Guarda el valor de la FO calculado en estado i (Con Bess)
        PARAM.loc[i,'SAIDI_Bi']=SAIDIi         #Guarda el SAIDI calculado en estado i (Con Bess)
        PARAM.loc[i,'SAIFI_Bi']=SAIFIi         #Guarda el SAIFI calculado en estado i (Con Bess)
        PARAM.loc[i,'ENE_Bi']=ENSi             #Guarda el ENS calculado en estado i (Con Bess)
        PARAM.loc[i,'CT_Bi']=CTi               #Guarda el CT calculado en estado i (Con Bess)
        PARAM.loc[i,'RG_Bi']=RGi               #Guarda el RG calculado en estado i (Con Bess)
        PARAM_Bess.loc[i,'PW'] =PBessi         #Guarda el Potencia calculado en estado i (del Bess)
        PARAM_Bess.loc[i,'EW'] =EBessi         #Guarda el Energia calculado en estado i (del Bess)

        #print(PARAM)
        #print(PARAM_1)
        #print(PARAM_Bess)
    return PARAM, PARAM_1, PARAM_Bess, FOi, RGi

#FUNCIÓN -SimCasEstAB- SIMULACIÓN DE CASOS DE ESTUDIO PARA HALLAR RESULTADOS EN UNA BARRA ALEATORIA CON DEMANDA ALEATORIA Y FALLA ALEATORIA
def SimCasEstAB (PAGeo,PACost,FCase,tipBess, i): #i es el bus aleatorio
    PARAM_1=pd.DataFrame(columns=['BusN','SAIDI_Bi-1','SAIFI_Bi-1','ENE_Bi-1'],index=range(len(PAGeo))) #Parametros en estado i-1 (Normal sin Bess)
    PARAM=pd.DataFrame(columns=['BusN','FO','SAIDI_Bi','SAIFI_Bi','ENE_Bi','CT_Bi','RG_Bi'],index=range(len(PAGeo))) #Parametros en estado i con BESS
    PARAM_Bess=pd.DataFrame(columns=['BusN','PW','EW'],index=range(len(PAGeo))) #Parametros en estado i con BESS
    PARAM_1.loc[:,'BusN']=PAGeo.loc[:,'BUS']
    PARAM.loc[:,'BusN']=PAGeo.loc[:,'BUS']
    PARAM_Bess.loc[:,'BusN']=PAGeo.loc[:,'BUS']
    #------------------------------------------------------------------------------
    #Crea las fallas y demandas del circuito en cada iteración
    #------------------------------------------------------------------------------
    exec(open("GF_Geneador_de_fallas.py").read())
    exec(open("GD_Generador_de_Demanda.py").read())
    #------------------------------------------------------------------------------
    #Definir Modelo para una barra
    #------------------------------------------------------------------------------
    print('Crea Caso de estudio ')
    #--------------------------------------------------------------------------
    #Consultar Varables
    #--------------------------------------------------------------------------
    BusPyo=PAGeo.loc[i,'BUS']                 #Extrae el bus de la lista de parametros
    IndyPE= ConsulIDX (BusPyo)                #Calcula el SAIDI, SAIFI, ENS PotenciaP y Energia del BESS
    INDICESi=IndyPE[0]                        #Asigana los indices calculados segun el bus escogido
    PE=IndyPE[1]                              #Calcula la potencia ye energia del BESS segun el bus escogido
    INDICESi_1=IndyPE[2]                      #Asigana los indices calculados segun el bus escogido
    #--------------------------------------------------------------------------
    #Calculo de indices de referencia i-1 (Normal sin Bess)
    #--------------------------------------------------------------------------
    PARAM_1.loc[i,'SAIDI_Bi-1']=INDICESi_1[0]     #Consulta y guarda el SAIDI calculado en estado i-1 (Normal sin Bess)
    PARAM_1.loc[i,'SAIFI_Bi-1']=INDICESi_1[1]     #Consulta y guarda el SAIFI calculado en estado i-1 (Normal sin Bess)
    PARAM_1.loc[i,'ENE_Bi-1']=INDICESi_1[2]       #Consulta y guarda el ENS calculado en estado i-1 (Normal sin Bess)
    #--------------------------------------------------------------------------
    #Establecer Indices
    #--------------------------------------------------------------------------
    a=FCase[0]   #Factor SAIDIP
    b=FCase[1]   #Factor SAIFI
    c=FCase[2]   #Factor ENS
    d=FCase[3]   #Factor COSTO
    e=FCase[4]   #Factor Elevación Geografica
    f=FCase[5]   #Factor Acsecibilidad
    g=FCase[6]   #Factor de Locación Urbano/Rural
    h=FCase[7]   #Factor de espacio
    l=FCase[8]   #Factor de Forestal
    #--------------------------------------------------------------------------
    #Consultar los indicesa calculados
    #--------------------------------------------------------------------------
    SAIDIi=float(INDICESi[0])                       #Consulta el SAIDI calculado
    SAIFIi=float(INDICESi[1])                       #Consulta el SAIFI calculado
    ENSi=float(INDICESi[2])                         #Consulta el ENS calculado
    EBessi=float(PE[0])                             #Consulta el Energia del Bess calculada
    PBessi=float(PE[1])                             #Consulta el Potencia del Bess calculada
    #--------------------------------------------------------------------------
    #Calculo de Parametro de Costo Asociado del Bess
    #--------------------------------------------------------------------------
    CTi=(PBessi*PACost.loc[tipBess,'Ppot'])+(EBessi*PACost.loc[tipBess,'Eenergy'])+(PBessi*PACost.loc[tipBess,'OM'])
    #--------------------------------------------------------------------------
    #Función Objetivo
    #--------------------------------------------------------------------------
    FOi=a*SAIDIi+b*SAIFIi+c*ENSi+d*CTi
    #--------------------------------------------------------------------------
    #Calculo de Parametro de Riesgo Geografico
    #--------------------------------------------------------------------------
    RGi=(e*PAGeo.loc[i,'NorC_Eleva'])+(f*PAGeo.loc[i,'NorC_Via'])+(g*PAGeo.loc[i,'NorC_Localiza'])+(h*PAGeo.loc[i,'NorC_Espacio'])+(l*PAGeo.loc[i,'NorC_Forestal'])
    #--------------------------------------------------------------------------
    #Almacenamiento de Variables, FO y Parametros
    #--------------------------------------------------------------------------
    PARAM.loc[i,'FO']=FOi                     #Guarda el valor de la FO calculado en estado i (Con Bess)
    PARAM.loc[i,'SAIDI_Bi']=SAIDIi            #Guarda el SAIDI calculado en estado i (Con Bess)
    PARAM.loc[i,'SAIFI_Bi']=SAIFIi            #Guarda el SAIFI calculado en estado i (Con Bess)
    PARAM.loc[i,'ENE_Bi']=ENSi                #Guarda el ENS calculado en estado i (Con Bess)
    PARAM.loc[i,'CT_Bi']=CTi                  #Guarda el CT calculado en estado i (Con Bess)
    PARAM.loc[i,'RG_Bi']=RGi                  #Guarda el RG calculado en estado i (Con Bess)
    PARAM_Bess.loc[i,'PW'] =PBessi            #Guarda el Potencia calculado en estado i (del Bess)
    PARAM_Bess.loc[i,'EW'] =EBessi            #Guarda el Energia calculado en estado i (del Bess)

    #print(PARAM)
    #print(PARAM_1)
    #print(PARAM_Bess)
    return PARAM, PARAM_1, PARAM_Bess

#FUNCIÓN -CreCROM- CREA EL LISTADO DE CROMOSOMAS DE TAMAÑO ESPECIFICO DEPENDE DEL NUMERO DE GENOMAS Y CROMOSOMAS
def CreCROM(Ng,Nc, PGeo_CR, PCost_CR, CasoE1_CR, tipBessE_CR):  # Ng:Numero de Genomas en un Cromosoma, Nc:Numero de cromosomas en la población
    GenPob=Ng*Nc #Numero de genomas de la población
    Cromi=pd.DataFrame(columns=['BusIdX','BusN','FO','SAIDI_Bi','SAIFI_Bi','ENE_Bi','CT_Bi','RG_Bi','PW','EW','SAIDI_Bi-1','SAIFI_Bi-1','ENE_Bi-1'],index=range(GenPob))  #Crea La estructura del Genoma
    #------------------------------------
    #Generador de Cromosomas
    #------------------------------------
    for i in range(GenPob):
        Con=0 #Condicional para evitar valores de potencia y costo cero (Ocurre cuando el switch fallado no desconecta potencia)
        while Con==0:
            Ba=random.randint(1, len(PGeo)-1)                                     #Escoge aleatoreamaente un barra de la lista PGeo !!!!!!!!!!
            ResTm=SimCasEstAB(PGeo_CR, PCost_CR, CasoE1_CR, tipBessE_CR, Ba)    #Calculo el Modelo de Optimización para i e i-1
            Rst1=ResTm[0]   #Asigana Parametros Calculados en i (Respuesta temporal 1)
            Rst2=ResTm[1]   #Asigana Parametros Calculado si-1  (Respuesta temporal 1)
            Rst3=ResTm[2]   #Asigana Parametros Calculados de Pot y Ener en i  (Respuesta temporal 1)
            #Asigna  y organiza los resualtados de calclo del Bus en el Cromosoma->
            Cromi.loc[i,'BusIdX']=Ba
            Cromi.loc[i,'BusN']=Rst1.loc[Ba,'BusN']
            Cromi.loc[i,'FO']=Rst1.loc[Ba,'FO']
            Cromi.loc[i,'SAIDI_Bi']=Rst1.loc[Ba,'SAIDI_Bi']
            Cromi.loc[i,'SAIFI_Bi']=Rst1.loc[Ba,'SAIFI_Bi']
            Cromi.loc[i,'ENE_Bi']=Rst1.loc[Ba,'ENE_Bi']
            Cromi.loc[i,'CT_Bi']=Rst1.loc[Ba,'CT_Bi']
            Cromi.loc[i,'RG_Bi']=Rst1.loc[Ba,'RG_Bi']
            Cromi.loc[i,'PW']=Rst3.loc[Ba,'PW']
            Cromi.loc[i,'EW']=Rst3.loc[Ba,'EW']
            Cromi.loc[i,'SAIDI_Bi-1']=Rst2.loc[Ba,'SAIDI_Bi-1']
            Cromi.loc[i,'SAIFI_Bi-1']=Rst2.loc[Ba,'SAIFI_Bi-1']
            Cromi.loc[i,'ENE_Bi-1']=Rst2.loc[Ba,'ENE_Bi-1']
            Con=Rst1.loc[Ba,'CT_Bi']
    return Cromi

#FUNCIÓN -CrePOB- CREA LA POBLACIÓN DE CONSULTA BASADO EN EL NUMERO DE CROMOSOMAS Y GENOMAS ESPECIFICADOS ()
def CrePOB(Ncp,Ngp,TGen):  #TGen: Tabla de Genomas
    TGen=TGen.reset_index()
    Pobi=pd.DataFrame(columns=range(Ngp),index=range(Ncp))  #Crea un dataframe de NcxNg columnas para lamecenar la población de resulyados de Funciones objetivo
    PoBus=pd.DataFrame(columns=range(Ngp),index=range(Ncp)) #Crea un dataframe de NcxNg columnas para lamecenar la población de Buses o Yaves
    PoBIdx=pd.DataFrame(columns=range(Ngp),index=range(Ncp)) #Crea un dataframe de NcxNg columnas para lamecenar la población de indices de los Buses
    PoBIX= pd.DataFrame(columns=range(Ngp),index=range(Ncp))
    for i in range(Ncp):  #Recorre las filas del dataFrame población y Bus
        if i<1:                                 #Condicional para llenado del primer cromosoma (fila)
            for j in range(Ngp):
                Pobi.iloc[i,j]=TGen.loc[j,'FO']
                PoBus.iloc[i,j]=TGen.loc[j,'BusN']
                PoBIdx.iloc[i,j]=TGen.loc[j,'BusIdX']
                PoBIX.iloc[i,j]=TGen.loc[j,'index']
        else:                                       #Condicional para llenado de los cromosomas siguientes (filas)
            ctGen= i*Ngp #Contador Genomas / recorre la tabla de genomas creados en PI (Población inicial) dependiendo el numero de genomas en cada cromosoma
            for j in range(Ngp):
                Pobi.iloc[i,j]=TGen.loc[ctGen,'FO']
                PoBus.iloc[i,j]=TGen.loc[ctGen,'BusN']
                PoBIdx.iloc[i,j]=TGen.loc[ctGen,'BusIdX']
                PoBIX.iloc[i,j]=TGen.loc[ctGen,'index']
                ctGen=ctGen+1   #Suma el siguiente consecutivo para recorrer la tabla genomas creados de PI
    #time.sleep(1)
    return Pobi, PoBus, PoBIdx, PoBIX #,TGen

#FUNCIÓN -FunFIT- EVALUA LOS CROMOSOMAS CON MENORES FUNCION OBJETIVO DE LA POBALCIÓN INICIAL
def FunFIT(Pob,PobB,PobBi,PobBix, Ncp, Ngp):
    Eval=pd.DataFrame(columns=range(Ngp),index=range(Ncp))  #Crea DataFrame Vacio con las dimenciones d ela población inicial
    EvalT=pd.DataFrame(columns=range(Ngp),index=range(Ncp)) #Crea DataFrame Vacio con las dimenciones d ela población inicial
    EvalT['Max']=np.nan #Agraga columNa vacia para hallar maximos
    EvalT['Sum']=np.nan #Agraga columNa vacia para hallar suma de maximos
    EvalT['IndxO']=np.nan #Agraga columNa vacia para hallar indices origeniles
    for i in range(len(Pob)): #Recorrela población de FO y halla lo svalores maximos por cromosoma
        EvalT.loc[i,'Max']=max(Pob.loc[i,:])  #Asigna el valor maximo ala poscición MAX en el DTF
        EvalT.loc[i,'IndxO']=i #Guarde el indice
    for i in range(len(Pob)): #Calcula la diferencia entre el maximo de cada cromosoma y cada geneoma de cada cromosoma
        for j in range(len(Pob)):
            Eval.loc[i,j]=EvalT.loc[i,'Max']-Pob.loc[i,j]#Calcula las diferencias y las carga en el DataFrame Temporal
            EvalT.loc[i,j]=Eval.loc[i,j]  #Asigna la diferencia al dataframe Permanaete
    EvalT['Sum']=Eval.sum(axis=1)  #Suma las diferencias de cada cromosoma
    Ord=EvalT.sort_values('Sum',ascending=False) #Ordena las sumas de las distancias de cada cromosoma de mayor a menor en el DataFrame
    if len(Ord)==1:
        Pob_P=Pob.filter(items = Ord.loc[:,'IndxO'], axis=0) #Filtra la población basada en el criterio de la suma de diferencias maximas
        PobB=PobB.filter(items = Ord.loc[:,'IndxO'], axis=0) #Filtra la población basada en el criterio de la suma de diferencias maximas
        PobBi=PobBi.filter(items = Ord.loc[:,'IndxO'], axis=0) #Filtra la población basada en el criterio de la suma de diferencias maximas
        PobBix=PobBix.filter(items = Ord.loc[:,'IndxO'], axis=0) #Filtra la población basada en el criterio de la suma de diferencias maximas
        pass
    else:
        Max=max(Ord['Sum'])-max(Ord['Sum'])*0.3 #Crea el limite minimo para aceptación basado en el valor maximo de suma
        if Max<0:
            Max=Max*-1
        My1 = Ord[Ord['Sum'] >= Max] #Filtra las sumas de las diferencias entre el maximo y la PI mayores a 1
        Pob_P=Pob.filter(items = My1.loc[:,'IndxO'], axis=0) #Filtra la población basada en el criterio de la suma de diferencias maximas
        PobB=PobB.filter(items = My1.loc[:,'IndxO'], axis=0) #Filtra la población basada en el criterio de la suma de diferencias maximas
        PobBi=PobBi.filter(items = My1.loc[:,'IndxO'], axis=0) #Filtra la población basada en el criterio de la suma de diferencias maximas
        PobBix=PobBix.filter(items = My1.loc[:,'IndxO'], axis=0) #Filtra la población basada en el criterio de la suma de diferencias maximas
    return Pob_P, PobB, PobBi, PobBix

#FUNCIÓN -CruCP- CREA UNA NUEVA POBLACIÓN CRUZANDO LA INICIAL BASADA EN EL PUNTO MEDIO
def CruCP (Pob,PobB,PobBi,PobBix,Ng):
    NgM=int(Ng/2)
    Pob=Pob.reset_index()
    PobB=PobB.reset_index()
    PobBi=PobBi.reset_index()
    PobBix=PobBix.reset_index()
    LonP=len(Pob) #Dimeción de la población
    if LonP == 1:  #Si la población es de un solo cromosoma, asigna el cromosola y rompe la etapa de cruce
        print('Población de un Cromososma')
        VecSel=0
        PobC=Pob
        PobBC=PobB
        PobBiC=PobBi
        PobBixC=PobBix
        pass
    else:
        PobC=pd.DataFrame(columns=range(Ng),index=range(LonP)) #Creación de Población vacia despues del cruce
        PobBC=pd.DataFrame(columns=range(Ng),index=range(LonP)) #Creación de Población vacia despues del cruce
        PobBiC=pd.DataFrame(columns=range(Ng),index=range(LonP)) #Creación de Población vacia despues del cruce
        PobBixC=pd.DataFrame(columns=range(Ng),index=range(LonP)) #Creación de Población vacia despues del cruce
        #Crea las conbinaciones aleatorias entre Cromosomas de la población
        VecSel=np.zeros((LonP,2)) #Crea una matriz de dos columnas con ceros
        Band=np.zeros(LonP) #Bandera que previene Mitosis
        #Prevención de Mitosis
        while (sum(Band)!=LonP):  #Como no se pueden crusar un solo cromososma, esta iteración asegura que las conbinaciones sean de diferentes cromososmasa, se repite si la suma!= de LonP
            for i in range(LonP): #LLena la matriz con los indices del dataframe aleatoreamente.
                for j in range(2):
                    VecSel[i,j]=random.randint(0, LonP-1)
            for i in range(len(VecSel)):  #Recorre la matriz de cruce
                if  VecSel[i,0]==VecSel[i,1]: #Si los numeros en una combinación son iguales asigna cero
                    Band[i]=0
                else:
                    Band[i]=1 #Si los numeros en una combinación son iguales asigna uno
        #Cruce de cormosomas Basado en la Matriz de cruce
        for i in range(len(PobC)):
            for j in range(NgM):
                PobC.loc[i,j]=Pob.loc[int(VecSel[i,0]),j] #Asigna el lado Izquierdo de la población
                PobBC.loc[i,j]=PobB.loc[int(VecSel[i,0]),j]
                PobBiC.loc[i,j]=PobBi.loc[int(VecSel[i,0]),j]
                PobBixC.loc[i,j]=PobBix.loc[int(VecSel[i,0]),j]
        for i in range(len(PobC)):
            for j in range(NgM,Ng):
                PobC.iloc[i,j]=Pob.loc[int(VecSel[i,1]),j] #Asigna el lado Derecho de la población
                PobBC.iloc[i,j]=PobB.loc[int(VecSel[i,1]),j] #Asigna el lado Derecho de la población
                PobBiC.iloc[i,j]=PobBi.loc[int(VecSel[i,1]),j] #Asigna el lado Derecho de la población
                PobBixC.iloc[i,j]=PobBix.loc[int(VecSel[i,1]),j] #Asigna el lado Derecho de la población
    return VecSel, PobC, PobBC, PobBiC, PobBixC

#FUNCIÓN -- MODIFICA LA POBLACIÓN CRUZADA CREANDO UNA MUTACIÓN ALEATORIA
def MutCP (PobM,PobBM,PobBiM, PobBixM, Ng,Nc, PGeoM, PCostM, CasoE1M, tipBessEM, LisG):
    LonP=len(PobM) #Longitud de la población
    indj=random.randint(0, LonP-1) #Indice fila aleatorio de la población
    indi=random.randint(0, Ng-1) #Indice columna aleatorio de la población
    LisGenM=CreCROM(1,1, PGeoM, PCostM, CasoE1M, tipBessEM) #Crea lista de un solo genoma
    MutP=CrePOB(1, 1,LisGenM) #Crea el genoma
    print('Genoma Creado')
    print('')
    GMutFO=MutP[0] #Extrae el valor de Función objetivo del Genoma
    GMutB=MutP[1] #Extrae el nombre de la Barra
    GMutBi=MutP[2] #Extare el indice general de la barra
    #Asigna el genoma a la población cruzada
    PobM.loc[indj,indi]=GMutFO.loc[0,0] #Asigna Valor función Objetivo
    PobBM.loc[indj,indi]=GMutB.loc[0,0] #Asigna el nombre del Bus
    PobBiM.loc[indj,indi]=GMutBi.loc[0,0] #Asigna el inidice general del Bus
    #Indc=PobBixM.loc[int(indj),int(indi)] #Indice de la barra en el listado inicial de la población
    #print('Inidces Mat GEN')
    #print(indj)
    #print(indi)
    #print('Indice LisGen')
    #print(Indc)
    LisG=LisG.append(LisGenM,ignore_index=True)
    return  PobM, PobBM, PobBiM, PobBixM, LisG

#==============================================================================
# ALGORITMO DE OPTIMIZACIÓN GENETICO
#==============================================================================
#------------------------------------------------------------------------------
#Importación de Parametros
#------------------------------------------------------------------------------
PGeo=pd.read_csv('4.Opti_Param/CriterioGeo.csv',sep=";") #Parametros Geometricos
PCost=pd.read_csv('4.Opti_Param/BessCost.csv',sep=";")   #Parametrosd e costso
PFacto=pd.read_csv('4.Opti_Param/Factores.csv',sep=";")   #Factores para estudios
#------------------------------------------------------------------------------
#Declara los casos de estudio
#------------------------------------------------------------------------------
CasoE1=PFacto.loc[:,'Estudio1']  #Extrae del dataframe los factores predefinidos
CasoE2=PFacto.loc[:,'Estudio2']  #Extrae del dataframe los factores predefinidos
CasoE3=PFacto.loc[:,'Estudio3']  #Extrae del dataframe los factores predefinidos
CasoE4=PFacto.loc[:,'Estudio4']  #Extrae del dataframe los factores predefinidos
tipBessE=4    #Tipo de Bess a evaluar: 0: Lead Acid, 1: NaS, 2: ZnBr, 3:Vanadium Redox, 4: Lithium IOM
CasoE=CasoE1 #Asigne el caso que desea DSimular
#Crear un DF de resultados de la Prueba
DFMsol=pd.DataFrame(columns=['BusIdX','BusN','FO','SAIDI_Bi','SAIFI_Bi','ENE_Bi','CT_Bi','RG_Bi','PW','EW','SAIDI_Bi-1','SAIFI_Bi-1','ENE_Bi-1'])#,index=range(1))
#------------------------------------------------------------------------------
#INICIA BUCLE DE ESTUDIO DE PRUEBA
#------------------------------------------------------------------------------
for i in range(100):
    #------------------------------------------------------------------------------
    #Población Inicial
    #------------------------------------------------------------------------------ ***********DEBE INICIAR EL BUCLE DE MUESTRA #Iniciar bucle de iteraciones
    print('==============================')
    print('Creación de Población Inicial')
    print('==============================')
    NGeno=2 #Numero de Genomas en un Cromosoma (Numero Par)
    NCrom=2 #Numero de Cromosomas en la población (Numero Par)
    LisGen=CreCROM(NGeno,NCrom, PGeo, PCost, CasoE, tipBessE) #listado de Genomas Simulados Población inicial (relación de tiempo 4x4 en seis minutos)
    LisGen.to_excel('7. Poblaciones/Poblacion_Var.xlsx') #Exportación Población
    LiGenM=pd.DataFrame(columns=['BusIdX','BusN','FO','SAIDI_Bi','SAIFI_Bi','ENE_Bi','CT_Bi','RG_Bi','PW','EW','SAIDI_Bi-1','SAIFI_Bi-1','ENE_Bi-1'],index=range(1))  #Listado de genomas Mutados
    PobIn=CrePOB(NCrom, NGeno,LisGen) #Crea la población en forma matricial basado en el listado de Genomas PI
    PoI_FO=PobIn[0]  #Aisla el dataframe de resultados de función Objetivo
    PoI_B=PobIn[1]   #Aisla el dataframe de nombres de Buses
    PoI_BI=PobIn[2]  #Aisla el dataframe de indices del modelo de Buses
    PoI_Bix=PobIn[3] #Aisla el dataframe de indicesdel DataFrame de Buses
    #------------------------------------------------------------------------------
    #Inicio de Bucle de Iteración Genetica
    #------------------------------------------------------------------------------
    print('==============================')
    print('Inicio de algoritmo Genetico')
    print('Procesando.....')
    print('==============================')
    Badr=0 #Bandera de numero de iteración
    LPob=len(PobIn) #Longitud población
    while (Badr<=100 and LPob>1):
        print('Iteración')
        print(Badr)
        #------------------------------------------------------------------------------
        #Evaluación de Población (Función Fitness)
        #------------------------------------------------------------------------------
        Pob_Fit= FunFIT(PoI_FO,PoI_B,PoI_BI,PoI_Bix, NCrom, NGeno)  #Extrae los cromosomas con mayor cantidad de genomas menores en la FO creando la selección de la población inicial
        #------------------------------------------------------------------------------
        #Cruce de la población (Punto Medio)
        #------------------------------------------------------------------------------
        Pob_Cru=CruCP(Pob_Fit[0], Pob_Fit[1], Pob_Fit[2], Pob_Fit[3], NGeno)
        #Condicional si el solo queda un cromososma en la población
        if len(Pob_Fit[0]!=1):
            #------------------------------------------------------------------------------
            #Mutación aleatoria posterios al cruce
            #------------------------------------------------------------------------------
            Pob_Mut=MutCP(Pob_Cru[1], Pob_Cru[2], Pob_Cru[3], Pob_Cru[4], NGeno, NCrom, PGeo, PCost, CasoE, tipBessE, LiGenM)
            PoI_FO=Pob_Mut[0]  #Aisla el dataframe de resultados de función Objetivo
            PoI_B=Pob_Mut[1]   #Aisla el dataframe de nombres de Buses
            PoI_BI=Pob_Mut[2]  #Aisla el dataframe de indices del modelo de Buses
            PoI_Bix=Pob_Mut[3] #Aisla el dataframe de indicesdel DataFrame de Buses
            LiGenM=Pob_Mut[4]
            #Asignación de respuestas
            CroResFO=PoI_FO
            CroResB= PoI_B
            CroResBi= PoI_BI
            CroResBix=PoI_Bix
        else:
            CroResFO=Pob_Fit[1]
            CroResB=Pob_Fit[2]
            CroResBi=Pob_Fit[3]
            CroResBix=Pob_Fit[4]
        if len(CroResFO)==1:
            break
        else:
            Badr=Badr+1
    #------------------------------------------------------------------------------
    #Seleción de Mejor Solución
    #------------------------------------------------------------------------------
    LisGenF=LisGen #Crea una capia del dataframe de la población Inicial
    LisGenMF=LiGenM #Crea una capia del dataframe de mutaciones
    #Limpiar Datos
    VecRFO=CroResFO.drop(['index'],axis=1) #Elimina columna adicional de indice
    VecRB=CroResB.drop(['index'],axis=1) #Elimina columna adicional de indice
    VecRBi=CroResBi.drop(['index'],axis=1) #Elimina columna adicional de indice
    VecRBix=CroResBix.drop(['index'],axis=1) #Elimina columna adicional de indice
    #VecRFO=CroResFO
    #VecRB=CroResB
    #VecRBi=CroResBi
    #VecRBix=CroResBix
    #Convertir a formato flotante
    VecRFO=VecRFO.astype('float64')
    CrdC=VecRFO.idxmin(axis=1) #Halla la coordanada de la Columna en el DataFrame con la FO minima
    CrdF=VecRFO.idxmin() #Halla la coordanada de la Fila en el DataFrame con la FO minima
    #Filtrar cromosoma para hallar el genoma minimo
    VFO=min(VecRFO.loc[0,:]) #min(CroResFO.loc[0,:]) #Valor minimo de la función objetivo

    if len(VecRFO)==1:
        print('Entro al IF')
        VBi=VecRBi.iloc[0,CrdC[0]] #Barra equivalanete al Valor minimo de la función objetivo hallada
        print('Barra')
        print(VBi)
        ResF = LisGen[LisGen['BusIdX'] == VBi] #Filtrar por la barra encontrada en el DF de la población Inicial
        print('Filtro')
        print(ResF)
        print('Filtro')
        print(len(ResF))
        if len(ResF)==0: #Si no encuentra la barra en el DF de la población inicial busca en el DF mutado
            print('Entro al IF IN')
            ResF = LisGenMF[LisGenMF['BusIdX'] == VBi] #Filtrar por la barra encontrada en el DF Mutado
    else:
        print('Entro al ELSE')
        VecFIL=pd.DataFrame(columns=['Min','iCol','iFil'],index=range(len(VecRFO))) #Creard Dataframe  para guardar el minimo de cada fila y los indices
        for i in range(len(VecRFO)):
            VecFIL.loc[i,'Min']=min(VecRFO.loc[i,:]) #Busca el minimo por fila y lo guarda en el DF en VecFIL
        for i in range(len(VecRFO)):
            for j  in range(NGeno):
                if  VecFIL.loc[i,'Min']==VecRFO.loc[i,j]: #Busca el minimo ene el vector y en la muestra seleccionado
                    VecFIL.loc[i,'iCol']=j #Agrega indice  colomna del minomo del vector
                    VecFIL.loc[i,'iFil']=i# Agrega indice  fila del minomo del vector
        VecFIL.sort_values(by='Min') #Ordena de menor a mayor
        VBi=VecRBi.iloc[VecFIL.loc[i,'iFil'], VecFIL.loc[i,'iCol']] #Encuentra la barra del valor minimo den la matriz
        ResF = LisGen[LisGen['BusIdX'] == VBi] #Filtrar por la barra encontrada en el DF de la población Inicial
        if len(ResF)!=1: #Si no encuentra la barra en el DF de la población inicial busca en el DF mutado
            ResF = LisGenMF[LisGenMF['BusIdX'] == VBi] #Filtrar por la barra encontrada en el DF Mutado
    #------------------------------------------------------------------------------
    #Almacenar la solución
    #------------------------------------------------------------------------------
    print('ALMACENA SOLUCIÓN')
    print(ResF)
    if len(ResF)==1: #Si hay una solución en e dataframe
        DFMsol=DFMsol.append(ResF) #Agregar las filas resultados al DF vacio
    #Si la Barra aparece mas de una vez en las poblaciones inicial y mutada se filtra por valor de FO minima
    else:#if len(ResF)!=1: #Si hay multiples soluciones
        ResF2=ResF
        ResF2=LisGen[LisGen['FO'] == VFO]
        ResF2=ResF2.reset_index()
        ResF2=ResF2.drop(['index'], axis=1)
        if len(ResF2)==0: #Si no encuentra la barra en el DF de la población inicial busca en el DF mutado
            ResF2 = LisGenMF[LisGenMF['FO'] == VFO]
            ResF2=ResF2.reset_index()
            ResF2=ResF2.drop(['index'], axis=1)
        #Si hay dos resutados identicos en los dos parametros (Barra y FO) toma aleatoreamnete cualquiera de los resultados
        print('SOLUCIÓN MULTIPLE')
        print(ResF)
        if len(ResF2)!=1:
            indFl=0#random.randint(0, len(ResF2)-1)
            print(indFl)
            ResF2=ResF2.loc[indFl,:]
        #------------------------------------------------------------------------------
        #Almacenar la solución
        #------------------------------------------------------------------------------
        DFMsol=DFMsol.append(ResF2) #Agregar las filas resultados al DF vacio
    print('==============================')
    print('Solución Obtenida')
    print('==============================')

#Exportar DF a csv o excel debe realizarse cuando finalice la prueba
DFMsol.to_excel('5. Resultados/Resultado_x-Caso_x.xlsx') #Exportación Población
print('==============================')
print('Resultados Exportados')
print('==============================')
#------------------------------------------------------------------------------
#Fin
#------------------------------------------------------------------------------
#Sigue tus pasiones y tu instinto, aunque te digan que no eres digno, porque los sueños se encuntran adelante en el camino, junto a tu familia y tus amigos (EO)